# Import packages

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


# Checks

In [ ]:
import paddle

paddle.utils.run_check()
print(paddle.__version__)


In [ ]:
%cd 1_Detection_Model
import torch
import utils
display = utils.notebook_init()
%cd ..


In [ ]:
!rm -rf /root/autodl-tmp/DT_SegNet/Detection_Output_KF
!rm -rf /root/autodl-tmp/DT_SegNet/Dataset/Segmentation_Input_KF
!rm -rf /root/autodl-tmp/DT_SegNet/Segmentation_Output_KF
!rm -rf /root/autodl-tmp/DT_SegNet/Output_KF

In [ ]:
# read YOLO format txt label, and perform dilation
def read_labels(label_path, img, dilation = 1.5):
    data = pd.read_csv(str(label_path), sep=" ", header=None,
                    names=["class", "x_center", "y_center", "width", "height", "proability"])
    img_x, img_y = img.size
    data['x0'] = ((data['x_center'] - dilation * data['width'] / 2) * img_x).astype("int")
    data['x1'] = ((data['x_center'] + dilation * data['width'] / 2) * img_x).astype("int")
    data['y0'] = ((data['y_center'] - dilation * data['height'] / 2) * img_y).astype("int")
    data['y1'] = ((data['y_center'] + dilation * data['height'] / 2) * img_y).astype("int")
    return data

In [ ]:
kf_ids = [0, 1, 2, 3, 4]
for kf_id in kf_ids:
    detection_model_path = f"/root/autodl-tmp/DT_SegNet/KF_res/{kf_id}.pt"
    !stat {detection_model_path}
    segmentation_model_path = f"/root/autodl-tmp/DT_SegNet/Segmentation_Model_Output_KF/{kf_id}/best_model/model.pdparams"
    !stat {segmentation_model_path}
    %cd 1_Detection_Model
    !python detect.py --project /root/autodl-tmp/DT_SegNet/Detection_Output_KF/{kf_id} --weights {detection_model_path} --img 1280 --source /root/autodl-tmp/DT_SegNet/Dataset/KFold/{kf_id}/val --line-thickness 2 --save-txt --save-conf --save-crop --conf-thres 0.475
    %cd ..
    detection_inference_exp_path = f"/root/autodl-tmp/DT_SegNet/Detection_Output_KF/{kf_id}/exp/"
    data_dir = Path(f'/root/autodl-tmp/DT_SegNet/Dataset/KFold/{kf_id}/val')
    label_dir = Path(detection_inference_exp_path) / 'labels'

    seg_output_dir = Path(f'/root/autodl-tmp/DT_SegNet/Dataset/Segmentation_Input_KF/{kf_id}/')
    seg_output_dir.mkdir(parents=True, exist_ok=True)
    for img_path in list(data_dir.glob('**/*.png')):
        print(f'Processing {str(img_path)}')

        # process cropped image
        img = Image.open(img_path)
        img = img.convert("L")
        # print(img.format, img.size, img.mode)
        labels = read_labels((label_dir / img_path.name).with_suffix('.txt'), img)
        with tqdm(total=len(labels)) as pbar:
            for index, r in labels.iterrows():
                if r.y0 == r.y1:
                    continue
                box = (r.x0, r.y0, r.x1, r.y1)
                region = img.crop(box)
                # region.show()
                croped_savepath = seg_output_dir / f'{img_path.stem}_{index}{img_path.suffix}'
                # print(croped_savepath, box)
                region.save(croped_savepath)
                pbar.update(1)
    %cd 3_Segmentation_Model
    !python predict.py --config configs/kfold/{kf_id}.yml --model_path {segmentation_model_path} --image_path /root/autodl-tmp/DT_SegNet/Dataset/Segmentation_Input_KF/{kf_id} --save_dir /root/autodl-tmp/DT_SegNet/Segmentation_Output_KF/{kf_id}/
    %cd ..
    data_dir = Path(f'/root/autodl-tmp/DT_SegNet/Dataset/KFold/{kf_id}/val')
    seg_output_dir = Path(f'/root/autodl-tmp/DT_SegNet/Segmentation_Output_KF/{kf_id}/pseudo_color_prediction')
    label_dir = Path(detection_inference_exp_path) / 'labels'
    output_root = Path(f'/root/autodl-tmp/DT_SegNet/Output_KF/{kf_id}')
    output_root.mkdir(exist_ok=True, parents=True)
    for img_path in list(data_dir.glob('*.png')):
        print(f'Processing {str(img_path)}')
        img = Image.open(img_path)
        img = img.convert("L")
        labels = read_labels((label_dir / img_path.name).with_suffix('.txt'), img)
        output = np.zeros_like(img)
        with tqdm(total=len(labels)) as pbar:
            for index, r in labels.iterrows():
                croped_path = seg_output_dir / f'{img_path.stem}_{index}{img_path.suffix}'
                if not croped_path.exists():
                    continue
                region = Image.open(croped_path)
                np_region = np.array(region)
                x0, x1, y0, y1 = int(r.x0), int(r.x1), int(r.y0), int(r.y1)
                for x in range(x1-x0):
                    for y in range(y1-y0):
                        y_out = y+y0
                        x_out = x+x0
                        y_out = max(0, y_out)
                        x_out = max(0, x_out)
                        y_out = min(output.shape[0]-1, y_out)
                        x_out = min(output.shape[1]-1, x_out)
                        output[y_out, x_out] += np_region[y, x] # add regions
                        # output[y_out, x_out] = np_region[y, x] # replace regions
                pbar.update(1)

        output[output>=1]=1
        data = np.array(output)
        new_data = data.astype('uint8')
        print(data.shape, new_data.shape, np.unique(new_data))
        plt.imshow(new_data, interpolation='nearest', cmap='Greys')
        # white background
        plt.show()
        np.save(output_root / f'{img_path.stem}.npy', new_data)
        Image.fromarray(
            new_data*255).save(output_root / f'{img_path.stem}.png')
